In [1]:
from huggingface_hub import login
from dotenv import load_dotenv

load_dotenv("../.env")
import os
HUGGING_FACE_TOKEN = os.getenv("HUGGING_FACE_TOKEN")
login(HUGGING_FACE_TOKEN)

from transformers import pipeline
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, AutoConfig, GenerationConfig

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: write).
Your token has been saved to /home/s448780/.cache/huggingface/token
Login successful


# Initialise model and tokenizer

In [2]:
model_id = "mistralai/Mistral-7B-Instruct-v0.1"

In [3]:
AutoConfig.from_pretrained(model_id, trust_remote_code=True)

MistralConfig {
  "_name_or_path": "mistralai/Mistral-7B-Instruct-v0.1",
  "architectures": [
    "MistralForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 1,
  "eos_token_id": 2,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 14336,
  "max_position_embeddings": 32768,
  "model_type": "mistral",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 8,
  "rms_norm_eps": 1e-05,
  "rope_theta": 10000.0,
  "sliding_window": 4096,
  "tie_word_embeddings": false,
  "torch_dtype": "bfloat16",
  "transformers_version": "4.43.4",
  "use_cache": true,
  "vocab_size": 32000
}

In [4]:
GenerationConfig.from_pretrained(model_id)

GenerationConfig {
  "bos_token_id": 1,
  "eos_token_id": 2
}

In [5]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)
model = AutoModelForCausalLM.from_pretrained(
        model_id, 
        quantization_config=bnb_config,
        use_cache = True,
        device_map = "auto", # requires accelerate
        torch_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_id,
                                         add_bos_token = True,
                                         padding_side = "left")
tokenizer.pad_token = tokenizer.eos_token

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [7]:
model

MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32000, 4096)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralSdpaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): MistralRotaryEmbedding()
        )
        (mlp): MistralMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): MistralRMSNorm()
        (post_attention_layernorm): MistralRMSNorm()
      )
    )

# Changing the chat template

In [19]:
inst_original_template = tokenizer.chat_template

In [23]:
tokenizer.chat_template = '''{%- for message in messages %}
    {%- if message['role'] == 'User' %}
        {{- 'User: ' + message['content'] + '\n\n'}}
    {%- elif message['role'] == 'System' %}
        {{- bos_token + 'System: ' + message['content'] + '\n\n'}}
    {%- endif %}
{%- endfor %}
{{- 'Assistant: '}}
'''

In [26]:
messages = [
    {
        "role": "System",
        "content": "You are a helpful chat assistant, always answer the question even if the context is not useful"
    },
    {
        "role": "User",
        "content": "Write a python fucntion to multiply 2 matrices"
    }
]

In [28]:
tokenizer.apply_chat_template(messages, tokenize=False)

'<s>System: You are a helpful chat assistant, always answer the question even if the context is not useful\n\nUser: Write a python fucntion to multiply 2 matrices\n\nAssistant: '

In [29]:
print(tokenizer.apply_chat_template(messages, tokenize=False))

<s>System: You are a helpful chat assistant, always answer the question even if the context is not useful

User: Write a python fucntion to multiply 2 matrices

Assistant: 


# Inference

In [37]:
inference = pipeline(
    model=model,
    tokenizer=tokenizer,
    task="text-generation",
    do_sample=False,
    repetition_penalty=1.1,
    return_full_text=False,
    max_new_tokens=1024
)

In [38]:
# prompt = "<s>Write a python fucntion to multiply 2 matrices"
prompt = tokenizer.apply_chat_template(messages, tokenize=False)

In [40]:
response = inference(prompt,
         tokenizer = tokenizer, # have to pass this for strop strings
         eos_token_id = tokenizer.eos_token_id,
         stop_strings = ["\n\nUser:"])

In [41]:
response

[{'generated_text': "\n\nHere's a Python function that multiplies two matrices:\n```python\ndef multiply_matrices(matrix1, matrix2):\n    # Check if the number of columns in matrix1 is equal to the number of rows in matrix2\n    if len(matrix1[0])!= len(matrix2):\n        return None\n    \n    # Initialize an empty matrix to store the result\n    result = [[0 for row in range(len(matrix2))] for col in range(len(matrix1))]\n    \n    # Multiply the matrices and store the result in the result matrix\n    for i in range(len(matrix1)):\n        for j in range(len(matrix2[0])):\n            for k in range(len(matrix2)):\n                result[i][j] += matrix1[i][k] * matrix2[k][j]\n            \n    return result\n```\nYou can use this function by passing two matrices as arguments. The function will return the product of the two matrices. If the number of columns in the first matrix is not equal to the number of rows in the second matrix, the function will return `None`."}]

In [42]:
print(response[0]["generated_text"])



Here's a Python function that multiplies two matrices:
```python
def multiply_matrices(matrix1, matrix2):
    # Check if the number of columns in matrix1 is equal to the number of rows in matrix2
    if len(matrix1[0])!= len(matrix2):
        return None
    
    # Initialize an empty matrix to store the result
    result = [[0 for row in range(len(matrix2))] for col in range(len(matrix1))]
    
    # Multiply the matrices and store the result in the result matrix
    for i in range(len(matrix1)):
        for j in range(len(matrix2[0])):
            for k in range(len(matrix2)):
                result[i][j] += matrix1[i][k] * matrix2[k][j]
            
    return result
```
You can use this function by passing two matrices as arguments. The function will return the product of the two matrices. If the number of columns in the first matrix is not equal to the number of rows in the second matrix, the function will return `None`.
